In [1]:
import os

os.chdir(os.getenv("HOME_DIR"))

import random

import onnxruntime as ort
import torch
import yaml
from dotenv import load_dotenv
from ultralytics import YOLO

from src.utils.metrics import fps

load_dotenv()

True

In [2]:
handle_model_yaml = "yolo8_baseline.yaml"
dataset_yaml = "bdd100k.yaml"
yaml_path = os.path.join(
    os.getenv("HOME_DIR"),
    "config",
    "models",
    handle_model_yaml,
)
with open(yaml_path, "r") as file:
    args = yaml.safe_load(file)

In [3]:
DATA_DIR = os.path.join(
    os.getenv("HOME_DIR"), "config", "datasets", dataset_yaml
)  # Default dataset_name.yaml or personal_dataset_name.yaml
IMG_SIZE = int(os.getenv("HEIGHT")), int(os.getenv("WIDTH"))
PROJECT_DIR = os.path.join(
    os.getenv("HOME_DIR"), "results", "models", args["project_results_name"]
)
TESTING_IMG_DIR = os.path.join(
    os.getenv("HOME_DIR"), "data", "processed", "images", "test"
)  # Testing images directory

# **Load models**

# **Base model**

In [4]:
onnx_path = os.path.join(PROJECT_DIR, "train", "weights", "best.onnx")
best_model_path = os.path.join(PROJECT_DIR, "train", "weights", "best.pt")

## **Measure model size**

In [5]:
model_size = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"Model size: {model_size:.2f} MB")

Model size: 11.62 MB


## **Load ONNX model on CPU**

In [6]:
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
image_path = os.path.join(
    TESTING_IMG_DIR,
    os.listdir(TESTING_IMG_DIR)[random.randint(0, len(os.listdir(TESTING_IMG_DIR)))],
)
result = fps(sess, image_path)
print(f"FPS on CPU (edge simulation): {round(result['fps'], 3)}")
print(f"Min time inference {round(min(result['times']) * 1000, 3)} ms")
print(
    f"Mean time inference {round(sum(result['times']) / len(result['times']) * 1000, 3)} ms"
)
print(f"Max time inference {round(max(result['times']) * 1000, 3)} ms")

FPS on CPU (edge simulation): 83.212
Min time inference 11.246 ms
Mean time inference 12.017 ms
Max time inference 15.954 ms


## **Get profile on CPU time**

In [7]:
best_model = YOLO(best_model_path, task="detect", verbose=True).to(torch.device("cpu"))
real_input_torch = torch.from_numpy(
    result["real_input"].transpose((0, 2, 3, 1))
).permute(0, 3, 1, 2)

with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
    best_model(real_input_torch)
print(prof.key_averages().table(sort_by="self_cpu_time_total"))


0: 480x480 18 cars, 41.7ms
Speed: 0.1ms preprocess, 41.7ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 480)
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   aten::mkldnn_convolution        48.70%      28.382ms        49.50%      28.849ms     480.812us            60  
                             aten::uniform_        16.99%       9.902ms        16.99%       9.902ms      86.863us           114  
                                   aten::mm         8.77%       5.111ms         8.80%       5.127ms      44.973us           114  
                                aten::copy_         4.69%       2.734ms         4.69%       2.7

# **Optimized model**

In [8]:
onnx_path = os.path.join(PROJECT_DIR, "optimized", "train3", "weights", "best.onnx")
best_model_path = os.path.join(PROJECT_DIR, "optimized", "train3", "weights", "best.pt")

## **Measure model size**

In [9]:
model_size = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"Model size: {model_size:.2f} MB")

Model size: 11.62 MB


## **Load ONNX model on CPU**

In [10]:
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
image_path = os.path.join(
    TESTING_IMG_DIR,
    os.listdir(TESTING_IMG_DIR)[random.randint(0, len(os.listdir(TESTING_IMG_DIR)))],
)
result = fps(sess, image_path)
print(f"FPS on CPU (edge simulation): {round(result['fps'], 3)}")
print(f"Min time inference {round(min(result['times']) * 1000, 3)} ms")
print(
    f"Mean time inference {round(sum(result['times']) / len(result['times']) * 1000, 3)} ms"
)
print(f"Max time inference {round(max(result['times']) * 1000, 3)} ms")

FPS on CPU (edge simulation): 78.125
Min time inference 11.151 ms
Mean time inference 12.8 ms
Max time inference 30.709 ms


## **Get profile on CPU time**

In [11]:
best_model = YOLO(best_model_path, task="detect", verbose=True).to(torch.device("cpu"))
real_input_torch = torch.from_numpy(
    result["real_input"].transpose((0, 2, 3, 1))
).permute(0, 3, 1, 2)

with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
    best_model(real_input_torch)
print(prof.key_averages().table(sort_by="self_cpu_time_total"))


0: 480x480 11 cars, 27.1ms
Speed: 0.7ms preprocess, 27.1ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 480)
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   aten::mkldnn_convolution        42.51%      17.685ms        43.35%      18.033ms     300.548us            60  
                             aten::uniform_        22.48%       9.352ms        22.48%       9.352ms      82.032us           114  
                                   aten::mm         6.47%       2.692ms         6.51%       2.707ms      23.746us           114  
                                aten::silu_         5.64%       2.347ms         5.64%       2.3